# Perception on a second model family: Pixtral-12B

Sixteenth notebook. The perception result (AUROC 0.835) is the one positive
finding that survived the 2026-08-09 retraction, and it rests on a single
model family. This is the attempt to change that.

Two previous attempts failed, for different reasons, and the gate below is
built to catch both before any real budget is spent:

| model | outcome |
|---|---|
| LLaVA-NeXT-7B | capability floor -- 3.0% transcription accuracy, only 9 correct items |
| InternVL3-8B | degenerate output -- 202/300 items at max entropy, AUROC collapsed 0.915 -> 0.556 under the artifact cut |

**Why Pixtral-12B.** The FERMAT paper benchmarks nine VLMs on this exact
dataset and finds Pixtral-124B among the strongest at reading the
handwriting. Pixtral-12B is the same family and architecture at a size that
fits a 40 GiB A100 in bf16. Mistral is genuinely independent of Qwen --
unlike MiniCPM-V and olmOCR, which are Qwen2-VL fine-tunes and would test
nothing.

**Interface, verified against the published chat template before writing any
code** (the lesson from InternVL3, which crashed on an unverified
`trust_remote_code` recipe):
- registers as `LlavaForConditionalGeneration`, no `trust_remote_code`;
- the template **does** support a system role, so this uses the same
  system+user shape as the Qwen runs rather than LLaVA's folded-in
  workaround -- better comparability with the 0.835 result;
- the system message must be a **plain string**: the template concatenates
  it directly (`"[INST]" + system_message`), so a content list would break;
- image chunks render as a bare `[IMG]` marker, so the image is passed
  separately to the processor rather than embedded in the message.

## Pre-registered gate (fixed before the run)

Process **50 items first**. The sample is always drawn at n=300 and merely
truncated, so the checkpoint carries straight over into the full run --
nothing is wasted if it passes.

```
transcription accuracy   >= 15%   else CAPABILITY FLOOR   (LLaVA scored 3.0%)
max-entropy fraction     <= 40%   else DEGENERATE          (InternVL3 hit 67.3%)
parse-failure rate       <= 25%   else FORMAT FAILURE
```

All three must pass. **No AUROC is reported at n=50** -- it cannot reach the
registered 30-item minority-class minimum, and printing one would only
invite it being quoted. For reference, Qwen-3B on the full run scored 39.3%
accuracy with 20.3% of items at max entropy.


In [1]:
# Install. Pixtral needs no special packages beyond a current transformers.
%pip install -q transformers accelerate datasets huggingface_hub bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 68.8 MB/s eta 0:00:00:00:0100:01


In [2]:
# Auth & code access. Identical to notebooks 12-15, including the
# sys.modules purge that makes a re-clone actually take effect.
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("Stored HF token does not start with 'hf_'; set RESET_TOKENS = True.")
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
for _n in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_n]

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy
import pilot.canonicalize
import pilot.plotting

print("pilot imported from:", os.path.dirname(pilot.__file__))


Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 13.3 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot imported from: /content/repo/pilot


In [3]:
# Model load. Pixtral-12B registers as LlavaForConditionalGeneration, so this
# is the standard path -- no trust_remote_code, which is what broke InternVL3.
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID = "mistral-community/pixtral-12b"
QUANTIZED = False

try:
    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    print(f"Loaded {MODEL_ID} in bfloat16 (full precision).")
except torch.cuda.OutOfMemoryError:
    print("bf16 did not fit -- falling back to 4-bit. This changes what is being "
          "measured; QUANTIZED=True is recorded in the results.")
    from transformers import BitsAndBytesConfig
    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                               bnb_4bit_compute_dtype=torch.bfloat16),
        device_map="auto", cache_dir=DRIVE_MODEL_CACHE,
    )
    QUANTIZED = True

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)
if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "left"   # required for batched generate

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GiB), quantized={QUANTIZED}")
if QUANTIZED:
    print("\nWARNING: 4-bit. The Qwen comparison (0.835) is bf16, so a dataset/"
          "model difference would be confounded with a precision difference. "
          "On a fresh runtime bf16 fits a 40 GiB A100 -- restart before trusting "
          "any cross-model comparison.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

Loaded mistral-community/pixtral-12b in bfloat16 (full precision).
GPU: NVIDIA A100-SXM4-40GB (39.5 GiB), quantized=False


In [4]:
# Sample. ALWAYS drawn at n=300 -- the same call every reference run used, so
# the comparison is apples-to-apples -- then truncated to PROCESS_N for the
# gate. Because the draw is identical, the gate items are a prefix of the full
# run and the checkpoint carries straight over. Set PROCESS_N = 300 and re-run
# the generation cell to continue; the first 50 are not regenerated.
import logging

import pilot.data

logging.basicConfig(level=logging.INFO)

SAMPLE_N = 300
SEED = 42
TARGET_ERROR_FRAC = 0.5
PROCESS_N = 300          # <-- the gate. Raise to 300 only after the gate passes.

full_sample = pilot.data.load_fermat_balanced(
    n=SAMPLE_N, seed=SEED, target_error_frac=TARGET_ERROR_FRAC)
sample = full_sample.select(range(PROCESS_N))
print(f"drawn {len(full_sample)} items, processing the first {len(sample)}")
print(f"has_error in this slice: {sum(bool(x) for x in sample['has_error'])}/{len(sample)}")


README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  481MB            

data/train-00001-of-00010.parquet: downloading bytes:           |  0.00B            

...

data/train-00008-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  493MB            

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2244 [00:00<?, ? examples/s]

drawn 300 items, processing the first 300
has_error in this slice: 150/300


In [5]:
# Adapter + pre-flight. Pixtral natively supports a system role, so this uses
# pilot.prompts.build_messages' system+user shape -- the same as the Qwen runs.
#
# Two Pixtral-specific details, both taken from the published chat template
# rather than assumed:
#   * the system message must be a PLAIN STRING (the template does
#     "[INST]" + system_message, which would fail on a content list);
#   * an image chunk renders as a bare [IMG] marker and carries no payload,
#     so the PIL image goes to the processor separately.
import pilot.prompts


def build_pixtral_messages(system_prompt: str, user_prompt: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},               # string, not a list
        {"role": "user", "content": [{"type": "image"},
                                     {"type": "text", "text": user_prompt}]},
    ]


def pixtral_inputs(system_prompt, user_prompt, image):
    text = processor.apply_chat_template(
        build_pixtral_messages(system_prompt, user_prompt), add_generation_prompt=True)
    return processor(text=text, images=[image], return_tensors="pt").to(model.device)


_item = sample[0]
_inp = pixtral_inputs(pilot.prompts.TRANSCRIPTION_SYSTEM_PROMPT,
                      pilot.prompts.TRANSCRIPTION_USER_PROMPT, _item["image"])
with torch.no_grad():
    _out = model.generate(**_inp, max_new_tokens=128, do_sample=False)
_txt = processor.batch_decode(_out[:, _inp["input_ids"].shape[1]:],
                              skip_special_tokens=True)[0]
print("Pre-flight OK. Greedy transcription sample:")
print(_txt[:400])
assert _txt.strip(), "empty output -- adapter is broken"
print("\nparses to:", repr(str(pilot.parsing.parse_transcription(_txt))[:120]))


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Pre-flight OK. Greedy transcription sample:
```latex
**Question:**
\[
\text{find the number of 4-letter words, with 0s without meaning, which can be formed out of the letters of the word ROSE, where the repetition of the letters is not allowed.}
\]

**Answer:**
\[
\text{There are many words as there are ways of filling in 4 vacant places --- by the 4 letters, keeping in mind that the repetition is not allowed. The first place can be filled 

parses to: '\\[\n\\text{There are many words as there are ways of filling in 4 vacant places --- by the 4 letters, keeping in mind that'


In [6]:
# Generation: K=5 per arm, batch-backoff ladder, per-item Drive checkpoint.
# The checkpoint is keyed by SAMPLE_N (300), NOT by PROCESS_N, so raising
# PROCESS_N to 300 resumes rather than restarts.
import gc
import json
import os
import time

from tqdm.auto import tqdm

K_TRANSCRIPTION = K_GRADING = 5
TEMP = 0.7
_LADDER = [5, 2, 1]
_state = {"i": 0}
META = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _batch(system_prompt, user_prompt, image, n, temperature):
    inputs = pixtral_inputs(system_prompt, user_prompt, image)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=True,
                             temperature=temperature, num_return_sequences=n)
    texts = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                   skip_special_tokens=True,
                                   clean_up_tokenization_spaces=False)
    del out, inputs
    gc.collect(); torch.cuda.empty_cache()
    return texts


def generate_k(system_prompt, user_prompt, image, n, temperature):
    texts, last = [], None
    while len(texts) < n:
        size = min(_LADDER[_state["i"]], n - len(texts))
        for attempt in range(3):
            try:
                texts += _batch(system_prompt, user_prompt, image, size, temperature)
                last = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                if _state["i"] + 1 < len(_LADDER):
                    _state["i"] += 1
                    print(f"  OOM at batch {size}; dropping to {_LADDER[_state['i']]}",
                          flush=True)
                    size = min(_LADDER[_state["i"]], n - len(texts))
                    continue
                raise
            except INFRA as exc:
                last = exc
                gc.collect(); torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last is not None:
            raise last
    return texts


CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
slug = MODEL_ID.split("/")[-1]
ckpt = (f"{CHECKPOINT_DIR}/pixtral_perc_{slug}_n{SAMPLE_N}_seed{SEED}"
        f"_k{K_TRANSCRIPTION}{'_4bit' if QUANTIZED else ''}.jsonl")

raw_results = []
if os.path.exists(ckpt):
    with open(ckpt) as f:
        raw_results = [json.loads(l) for l in f if l.strip()]
    valid = []
    for idx, e in enumerate(raw_results):
        if idx >= len(full_sample):
            break
        it = full_sample[idx]
        if not all(e["item"].get(k) == it[k] for k in META):
            print(f"checkpoint item {idx+1} mismatch; resuming there."); break
        if (len(e.get("transcription_samples_raw", [])) != K_TRANSCRIPTION
                or len(e.get("grading_samples_raw", [])) != K_GRADING):
            break
        valid.append(e)
    if len(valid) != len(raw_results):
        with open(ckpt, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
    raw_results = valid
    print(f"resuming from {len(raw_results)} completed items")

todo = [i for i in range(PROCESS_N) if i >= len(raw_results)]
if not todo:
    print(f"first {PROCESS_N} items already done.")
else:
    print(f"generating items {todo[0]+1}..{todo[-1]+1}", flush=True)
    with tqdm(total=len(todo) * (K_TRANSCRIPTION + K_GRADING),
              desc="pixtral", unit="sample") as pbar:
        for idx in todo:
            item = full_sample[idx]
            t0 = time.time()
            tr = generate_k(pilot.prompts.TRANSCRIPTION_SYSTEM_PROMPT,
                            pilot.prompts.TRANSCRIPTION_USER_PROMPT,
                            item["image"], K_TRANSCRIPTION, TEMP)
            pbar.update(K_TRANSCRIPTION)
            gr = generate_k(pilot.prompts.GRADING_SYSTEM_PROMPT,
                            pilot.prompts.GRADING_USER_PROMPT,
                            item["image"], K_GRADING, TEMP)
            pbar.update(K_GRADING)
            entry = {"item": {k: item[k] for k in META},
                     "transcription_samples_raw": tr, "grading_samples_raw": gr,
                     "quantized": QUANTIZED, "elapsed_seconds": time.time() - t0}
            raw_results.append(entry)
            with open(ckpt, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n"); f.flush()
            print(f"  item {idx+1}/{PROCESS_N}: {time.time()-t0:.1f}s", flush=True)

print(f"raw_results: {len(raw_results)} items")


resuming from 50 completed items
generating items 51..300


pixtral:   0%|          | 0/2500 [00:00<?, ?sample/s]

  item 51/300: 22.1s
  item 52/300: 25.0s
  item 53/300: 17.3s
  item 54/300: 19.8s
  item 55/300: 14.9s
  item 56/300: 27.2s
  item 57/300: 41.9s
  item 58/300: 29.4s
  item 59/300: 37.6s
  item 60/300: 41.0s
  item 61/300: 21.9s
  item 62/300: 28.0s
  item 63/300: 25.4s
  item 64/300: 28.7s
  item 65/300: 32.8s
  item 66/300: 35.2s
  item 67/300: 25.1s
  item 68/300: 20.6s
  item 69/300: 38.0s
  item 70/300: 37.5s
  item 71/300: 20.4s
  item 72/300: 17.4s
  item 73/300: 49.0s
  item 74/300: 24.1s
  item 75/300: 20.9s
  item 76/300: 27.7s
  item 77/300: 19.3s
  item 78/300: 33.9s
  item 79/300: 38.5s
  item 80/300: 24.2s
  item 81/300: 35.4s
  item 82/300: 25.2s
  item 83/300: 34.1s
  item 84/300: 38.5s
  item 85/300: 26.7s
  item 86/300: 27.1s
  item 87/300: 25.8s
  item 88/300: 27.8s
  item 89/300: 47.0s
  item 90/300: 40.9s
  item 91/300: 34.7s
  item 92/300: 20.2s
  item 93/300: 39.0s
  item 94/300: 26.8s
  item 95/300: 37.7s
  item 96/300: 21.0s
  item 97/300: 33.3s
  item 98/300

In [7]:
# THE GATE. Scores what exists and applies the three pre-registered checks.
# Deliberately prints NO AUROC while PROCESS_N < 300: 50 items cannot reach
# the registered 30-item minority-class minimum, and a number printed here
# would only invite being quoted.
import math

import pandas as pd

import pilot.canonicalize
import pilot.entropy
import pilot.parsing

GATE_MIN_ACCURACY   = 0.15   # LLaVA scored 0.030 -> capability floor
GATE_MAX_MAXENT     = 0.40   # InternVL3 hit 0.673 -> degenerate
GATE_MAX_PARSE_FAIL = 0.25

rows = []
for e in raw_results[:PROCESS_N]:
    it = e["item"]
    tp = [pilot.parsing.parse_transcription(s) for s in e["transcription_samples_raw"]]
    labels = [pilot.canonicalize.canonical_answer_label(t) for t in tp]
    gp = [pilot.parsing.parse_grading(s) for s in e["grading_samples_raw"]]
    glab = [None if d is None else str(d) for d in gp]
    maj, _ = pilot.entropy.majority_cluster(labels)
    gmaj, _ = pilot.entropy.majority_cluster(glab)
    rows.append({
        "orig_q": it["orig_q"], "pert_a": it["pert_a"], "has_error": it["has_error"],
        "handwriting_style": it["handwriting_style"], "image_quality": it["image_quality"],
        "perception_entropy": pilot.entropy.cluster_entropy(labels),
        "reasoning_entropy": pilot.entropy.cluster_entropy(glab),
        "transcription_correct": maj == pilot.canonicalize.canonical_answer_label(it["pert_a"]),
        "grading_correct": gmaj in {"0","1"} and int(gmaj) == int(it["has_error"]),
        "n_transcription_parse_failures": sum(1 for t in tp if t is None),
        "n_grading_parse_failures": sum(1 for d in gp if d is None),
        "all_transcription_samples_raw": e["transcription_samples_raw"],
        "all_grading_samples_raw": e["grading_samples_raw"],
        "model_id": MODEL_ID, "quantized": e["quantized"],
        "n_items": PROCESS_N, "k_transcription": K_TRANSCRIPTION, "k_grading": K_GRADING,
        "target_error_frac": TARGET_ERROR_FRAC,
    })
df = pd.DataFrame(rows)

acc      = df["transcription_correct"].mean()
maxent   = df["perception_entropy"].apply(lambda e: math.isclose(e, math.log(5))).mean()
parsefail= df["n_transcription_parse_failures"].sum() / (len(df) * K_TRANSCRIPTION)

print(f"n = {len(df)}\n")
print(f"  transcription accuracy   {acc:6.1%}   (gate >= {GATE_MIN_ACCURACY:.0%};  "
      f"Qwen-3B 39.3%, LLaVA 3.0%)")
print(f"  max-entropy fraction     {maxent:6.1%}   (gate <= {GATE_MAX_MAXENT:.0%};  "
      f"Qwen-3B 20.3%, InternVL3 67.3%)")
print(f"  parse-failure rate       {parsefail:6.1%}   (gate <= {GATE_MAX_PARSE_FAIL:.0%})")
print()

checks = {
    "accuracy":      acc       >= GATE_MIN_ACCURACY,
    "not degenerate": maxent   <= GATE_MAX_MAXENT,
    "parses":        parsefail <= GATE_MAX_PARSE_FAIL,
}
for name, ok in checks.items():
    print(f"    {'PASS' if ok else 'FAIL'}  {name}")
print()
if all(checks.values()):
    print("GATE PASSED. Set PROCESS_N = 300 in the sample cell and re-run it plus")
    print("the generation cell; the first 50 items resume from the checkpoint.")
else:
    failed = [n for n, ok in checks.items() if not ok]
    print(f"GATE FAILED on: {', '.join(failed)}.")
    print("Do NOT scale to 300. Report as a capability/degeneracy gate, the same")
    print("way LLaVA-NeXT and InternVL3 were, and save this CSV as the evidence.")

if PROCESS_N < 300:
    print("\n(No AUROC printed: 50 items cannot clear the 30-item minority-class")
    print(" minimum, so any value here would be uninterpretable.)")

print("\nRead these before trusting the verdict -- is the model reading the page,")
print("or emitting plausible-looking boilerplate?")
for i in range(min(3, len(df))):
    print("=" * 72)
    print("GT :", str(df.iloc[i]["pert_a"])[:150].replace("\n", " "))
    print("s0 :", str(df.iloc[i]["all_transcription_samples_raw"][0])[:300].replace("\n", " "))


n = 300

  transcription accuracy    41.7%   (gate >= 15%;  Qwen-3B 39.3%, LLaVA 3.0%)
  max-entropy fraction      17.0%   (gate <= 40%;  Qwen-3B 20.3%, InternVL3 67.3%)
  parse-failure rate         2.7%   (gate <= 25%)

    PASS  accuracy
    PASS  not degenerate
    PASS  parses

GATE PASSED. Set PROCESS_N = 300 in the sample cell and re-run it plus
the generation cell; the first 50 items resume from the checkpoint.

Read these before trusting the verdict -- is the model reading the page,
or emitting plausible-looking boilerplate?
GT :  There are as many words as there are ways of filling in 4 vacant places \_\_\_\_ by the 4 letters, keeping in mind that the repetition is not allowed
s0 : ```latex **Question:** \[ \text{Find the number of 4-letter words, with 08 without meaning, which can be formed out of the letters of the word ROSE, where the repetition of the letters is not allowed.} \]  **Answer:** \[ \text{There are many words as there are ways of filling in 4 vacant places ----

In [8]:
# Save. Drive first, then repo + push -- Drive is the source of truth because
# pushes from Colab 403 routinely here (see notebook 15).
import os
import subprocess
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
tag = "gate" if PROCESS_N < 300 else "full"
csv_name = (f"pixtral_perception_{tag}_n{PROCESS_N}_"
            f"{'4bit_' if QUANTIZED else ''}pixtral-12b_{timestamp}.csv")

drive_results = f"{PROJECT_DIR}/results"
os.makedirs(drive_results, exist_ok=True)
df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
df.to_csv(f"repo/results/{csv_name}", index=False)
print(f"Wrote repo/results/{csv_name} ({len(df)} rows)")

_REDACT = []


def git(*args):
    r = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    for s in _REDACT:
        if s:
            out = out.replace(s, "***")
    if r.returncode != 0 and out.strip():
        print(out.strip())
    return r


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
if git("commit", "-m", f"Add Pixtral-12B perception results: {csv_name}").returncode != 0:
    print("git commit failed -- CSV is safe on Drive.")

tok = (globals().get("GH_TOKEN") or "").strip()
_REDACT.append(tok)
pushed = False
if tok:
    url = REPO_URL.replace("https://", f"https://{tok}@")
    if git("fetch", url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
    pushed = git("push", url, "HEAD:main").returncode == 0
print("Pushed." if pushed else
      f"Push failed or skipped -- the CSV is on Drive at {drive_results}/{csv_name}, "
      "nothing is lost.")


Backup written to /content/drive/MyDrive/uncertainty-math-vlm/results/pixtral_perception_full_n300_pixtral-12b_20260809T211028Z.csv
Wrote repo/results/pixtral_perception_full_n300_pixtral-12b_20260809T211028Z.csv (300 rows)
remote: Permission to sepehrmaleki369/uncertainty-math-vlm.git denied to sepehrmaleki369.
fatal: unable to access 'https://github.com/sepehrmaleki369/uncertainty-math-vlm.git/': The requested URL returned error: 403
Push failed or skipped -- the CSV is on Drive at /content/drive/MyDrive/uncertainty-math-vlm/results/pixtral_perception_full_n300_pixtral-12b_20260809T211028Z.csv, nothing is lost.
